# 🌐 Nationality Detection — Experiments & Colab Testing

**Internship Task:** Nationality Detection with conditional attribute prediction

## Rules implemented:
| Nationality | Predictions Made |
|---|---|
| 🇮🇳 Indian | emotion + age + dress colour |
| 🇺🇸 American | emotion + age |
| 🌍 African | emotion + dress colour |
| 🌐 Other | nationality + emotion only |


In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
!pip install deepface opencv-python opencv-contrib-python scikit-learn \
             matplotlib seaborn pillow numpy tf-keras -q
print('✓ Dependencies installed')

In [ ]:
# ── CELL 2: Imports & setup ───────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os, sys
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
os.makedirs('outputs', exist_ok=True)
print('✓ Setup complete')

In [ ]:
# ── CELL 3: Download age model weights ───────────────────────────────────────
!python download_models.py

In [ ]:
# ── CELL 4: Test dress colour detection ──────────────────────────────────────
sys.path.insert(0, '.')
from utils.predictor import closest_colour_name, DRESS_COLOUR_PALETTE

# Test with pure RGB colours
test_colours = [
    ((255, 0,   0),   'Expected: Red'),
    ((0,   0,   255), 'Expected: Blue or Navy Blue'),
    ((0,   128, 0),   'Expected: Green'),
    ((255, 255, 255), 'Expected: White'),
    ((0,   0,   0),   'Expected: Black'),
    ((139, 69,  19),  'Expected: Brown'),
    ((255, 165, 0),   'Expected: Orange'),
    ((128, 0,   128), 'Expected: Purple'),
]

print('Dress Colour Detection Test:')
print('-' * 40)
for rgb, note in test_colours:
    result = closest_colour_name(rgb)
    print(f'  RGB{rgb} → {result:15s}  ({note})')

In [ ]:
# ── CELL 5: Visualise colour palette ─────────────────────────────────────────
from config import DRESS_COLOUR_PALETTE

names = list(DRESS_COLOUR_PALETTE.keys())
rgbs  = list(DRESS_COLOUR_PALETTE.values())

fig, ax = plt.subplots(figsize=(14, 6))
for i, (name, (r, g, b)) in enumerate(DRESS_COLOUR_PALETTE.items()):
    col = i % 12
    row = i // 12
    rect = plt.Rectangle([col, -row], 1, 1,
                          color=(r/255, g/255, b/255))
    ax.add_patch(rect)
    text_col = 'white' if (r + g + b) < 380 else 'black'
    ax.text(col + 0.5, -row + 0.3, name, ha='center', va='center',
            fontsize=6.5, color=text_col, weight='bold')

ax.set_xlim(0, 12)
ax.set_ylim(-4, 1.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Dress Colour Palette (34 named colours)', fontsize=13, pad=10)
plt.tight_layout()
plt.savefig('outputs/colour_palette.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total colours: {len(DRESS_COLOUR_PALETTE)}')

In [ ]:
# ── CELL 6: Test on a real image ─────────────────────────────────────────────
# Upload your own image OR use the file uploader below
from google.colab import files
from utils.predictor import NationalityPredictor
import matplotlib.pyplot as plt
import cv2

print('Upload a face image...')
uploaded = files.upload()

predictor = NationalityPredictor()

for filename, data in uploaded.items():
    img_path = f'samples/{filename}'
    os.makedirs('samples', exist_ok=True)
    with open(img_path, 'wb') as f:
        f.write(data)

    print(f'\nAnalysing: {filename}...')
    result = predictor.analyse(img_path)

    # Print structured result
    print(f'\n{"═"*45}')
    print(f'  Nationality  : {result["nationality"]} ({result.get("confidence", 0):.1f}%)')
    print(f'  Task Group   : {result["task_group"].upper()}')
    print(f'  Emotion      : {result.get("emotion", "N/A")}')
    if result.get('age'):
        print(f'  Age          : ~{result["age"]} years')
    if result.get('dress_colour'):
        print(f'  Dress Colour : {result["dress_colour"]}')
    print(f'  Face Found   : {"Yes" if result["face_found"] else "No"}')
    print(f'{"═"*45}')

    # Show annotated image
    if result.get('annotated_image') is not None:
        rgb = cv2.cvtColor(result['annotated_image'], cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 6))
        plt.imshow(rgb)
        plt.axis('off')
        plt.title(f"{result['nationality']} · {result.get('emotion', '')}")
        plt.tight_layout()
        plt.savefig(f'outputs/result_{filename}', dpi=150)
        plt.show()

In [ ]:
# ── CELL 7: Verify conditional prediction rules ───────────────────────────────
from config import TASK_RULES, NATIONALITY_MAP

print('Prediction Rules per Nationality Group:')
print('='*52)
icons = {'indian': '🇮🇳', 'american': '🇺🇸', 'african': '🌍', 'other': '🌐'}

for group, rules in TASK_RULES.items():
    active = [k for k, v in rules.items() if v]
    print(f'  {icons.get(group, "")} {group.upper():12s} → {" + ".join(active)}')

print('='*52)
print('\nNationality Mapping (DeepFace label → display):')
for raw, display in NATIONALITY_MAP.items():
    print(f'  {raw:20s} → {display}')

In [ ]:
# ── CELL 8: Visualise prediction rules as matrix ─────────────────────────────
import pandas as pd
import seaborn as sns

groups = ['Indian 🇮🇳', 'American 🇺🇸', 'African 🌍', 'Other 🌐']
attrs  = ['Nationality', 'Emotion', 'Age', 'Dress Colour']

# 1 = predicted, 0 = not predicted
matrix = [
    [1, 1, 1, 1],  # Indian
    [1, 1, 1, 0],  # American
    [1, 1, 0, 1],  # African
    [1, 1, 0, 0],  # Other
]

df = pd.DataFrame(matrix, index=groups, columns=attrs)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    df, annot=True, fmt='d', cmap='YlGn',
    linewidths=1, linecolor='white',
    cbar=False, ax=ax,
    annot_kws={'size': 14}
)
ax.set_title('Conditional Prediction Matrix\n(1 = Predicted, 0 = Not Predicted)',
             fontsize=12, pad=12)
ax.set_xticklabels(attrs, fontsize=11)
ax.set_yticklabels(groups, fontsize=11, rotation=0)

# Replace 1/0 annotations with ✓/✗
for text in ax.texts:
    text.set_text('✓' if text.get_text() == '1' else '✗')
    text.set_fontsize(16)

plt.tight_layout()
plt.savefig('outputs/prediction_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/prediction_matrix.png')

In [ ]:
# ── CELL 9: DeepFace emotion + race demo (standalone) ────────────────────────
# This shows DeepFace working directly — good for your notebook submission
try:
    from deepface import DeepFace
    import cv2

    # Use the image you uploaded earlier
    img_files = [f for f in os.listdir('samples') if f.endswith(('.jpg', '.png', '.jpeg'))]

    if img_files:
        img_path = f'samples/{img_files[0]}'
        print(f'Running DeepFace on: {img_path}')

        analysis = DeepFace.analyze(
            img_path,
            actions=['race', 'emotion', 'age'],
            enforce_detection=False,
            silent=True
        )

        if isinstance(analysis, list):
            analysis = analysis[0]

        print('\nDeepFace Raw Output:')
        print(f'  dominant_race    : {max(analysis["race"], key=analysis["race"].get)}')
        print(f'  dominant_emotion : {analysis["dominant_emotion"]}')
        print(f'  age              : {analysis["age"]}')

        # Plot race confidence breakdown
        race_data = analysis['race']
        fig, ax = plt.subplots(figsize=(8, 4))
        bars = ax.barh(list(race_data.keys()), list(race_data.values()),
                       color=['#4ecca3' if v == max(race_data.values())
                              else '#a8dadc' for v in race_data.values()])
        ax.set_xlabel('Confidence (%)')
        ax.set_title('DeepFace Race/Ethnicity Confidence Scores')
        for bar, val in zip(bars, race_data.values()):
            ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}%', va='center', fontsize=9)
        plt.tight_layout()
        plt.savefig('outputs/race_confidence.png', dpi=150)
        plt.show()
    else:
        print('Upload an image first (Cell 6)')

except ImportError:
    print('DeepFace not installed — run Cell 1 first')

In [ ]:
# ── CELL 10: Download all outputs ────────────────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('nationality_outputs', 'zip', 'outputs')
files.download('nationality_outputs.zip')
print('Downloaded outputs.zip')